## DATA Ingestion

In [46]:
## Document STructure

from langchain_core.documents import Document

In [47]:
## Each chunk becomes a new Document  &&   Metadata is automatically copied to every chunk
## Metadata is very useful when we doing similarity search we can do filters

doc=Document(
    page_content="Main content of RAG",
    metadata={
        "source":"example.txt",
        "pages":20,
        "author":"bharat"
    }
)
doc

Document(metadata={'source': 'example.txt', 'pages': 20, 'author': 'bharat'}, page_content='Main content of RAG')

In [48]:
## Creating text file

import os
os.makedirs("../data/text_files",exist_ok=True)

In [49]:
sample_texts={
    "../data/text_files/python_intro.txt": """Intro of Python
Python is a high-level, interpreted programming language known for its simplicity and readability.
It supports multiple programming paradigms, including procedural, object-oriented, and functional programming.

Python is widely used in web development, data science, artificial intelligence, and automation.
Popular frameworks such as Flask and Django are used to build web applications.
Libraries like NumPy, Pandas, and Matplotlib are commonly used for data analysis and visualization.

In machine learning, Python provides libraries such as scikit-learn, TensorFlow, and PyTorch.
These libraries help developers build, train, and evaluate machine learning models efficiently.

Python is also widely used in natural language processing.
Tools like NLTK, spaCy, and Hugging Face Transformers enable text processing and language understanding tasks.

Because of its large ecosystem and active community, Python is one of the most preferred languages for beginners and professionals.
""",

    "../data/text_files/agriculture_ai.txt": """Agriculture AI
Artificial Intelligence is transforming agriculture by improving productivity and reducing risk.
AI systems can analyze soil nutrients, weather conditions, and crop images to support farmers in decision-making.

Crop disease detection systems identify plant diseases using leaf images.
Irrigation optimization systems suggest when and how much water should be supplied to crops.
These AI tools help farmers increase yield and reduce resource wastage.
"""
}
for filepath,content in sample_texts.items():
    with open(filepath,'w',encoding="utf-8") as f:
        f.write(content)
print("Sample Text Created")

Sample Text Created


In [185]:
### TextLoader

from langchain_community.document_loaders import TextLoader

loader=TextLoader("../data/text_files/python_intro.txt",encoding="utf-8")
document=loader.load()
print(document)
loader1=TextLoader("../data/text_files/agriculture_ai.txt",encoding="utf-8")
print(loader1.load())

[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Intro of Python\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nIt supports multiple programming paradigms, including procedural, object-oriented, and functional programming.\n\nPython is widely used in web development, data science, artificial intelligence, and automation.\nPopular frameworks such as Flask and Django are used to build web applications.\nLibraries like NumPy, Pandas, and Matplotlib are commonly used for data analysis and visualization.\n\nIn machine learning, Python provides libraries such as scikit-learn, TensorFlow, and PyTorch.\nThese libraries help developers build, train, and evaluate machine learning models efficiently.\n\nPython is also widely used in natural language processing.\nTools like NLTK, spaCy, and Hugging Face Transformers enable text processing and language understanding tasks.\n\nBecause of its large ecosystem and 

In [ ]:
### Directory Loader

from langchain_community.document_loaders import DirectoryLoader

dir_loader = DirectoryLoader(
    path="../data/text_files",
    glob="**/*.txt",
    loader_cls=TextLoader,
    loader_kwargs={'encoding': 'utf-8'},
    show_progress=False
)

documents = dir_loader.load()
documents



[Document(metadata={'source': '..\\data\\text_files\\agriculture_ai.txt'}, page_content='Agriculture AI\nArtificial Intelligence is transforming agriculture by improving productivity and reducing risk.\nAI systems can analyze soil nutrients, weather conditions, and crop images to support farmers in decision-making.\n\nCrop disease detection systems identify plant diseases using leaf images.\nIrrigation optimization systems suggest when and how much water should be supplied to crops.\nThese AI tools help farmers increase yield and reduce resource wastage.\n'),
 Document(metadata={'source': '..\\data\\text_files\\python_intro.txt'}, page_content='Intro of Python\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nIt supports multiple programming paradigms, including procedural, object-oriented, and functional programming.\n\nPython is widely used in web development, data science, artificial intelligence, and automation.\nPopular frameworks

In [236]:
### PDF loader

from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader

dir_loader = DirectoryLoader(
    path="../data/pdfs",
    glob="**/*.pdf",
    loader_cls=PyMuPDFLoader,
    show_progress=False
)

pdf_documents = dir_loader.load()
pdf_documents

[Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': '', 'creationdate': '2025-12-21T20:04:05+05:30', 'source': '..\\data\\pdfs\\i_.pdf', 'file_path': '..\\data\\pdfs\\i_.pdf', 'total_pages': 15, 'format': 'PDF 1.7', 'title': '1Intro.pmd', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2025-12-21T20:04:05+05:30', 'trapped': '', 'modDate': "D:20251221200405+05'30'", 'creationDate': "D:20251221200405+05'30'", 'page': 0}, page_content=''),
 Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': '', 'creationdate': '2025-12-21T20:04:05+05:30', 'source': '..\\data\\pdfs\\i_.pdf', 'file_path': '..\\data\\pdfs\\i_.pdf', 'total_pages': 15, 'format': 'PDF 1.7', 'title': '1Intro.pmd', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2025-12-21T20:04:05+05:30', 'trapped': '', 'modDate': "D:20251221200405+05'30'", 'creationDate': "D:20251221200405+05'30'", 'page': 1}, page_content=''),
 Document(metadata={'producer': 'Microsoft: Print To PDF', 'crea

In [ ]:
type(pdf_documents[0])

langchain_core.documents.base.Document

In [ ]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [239]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 4 PDF files to process

Processing: i_.pdf
  ✓ Loaded 15 pages

Processing: py_.pdf
  ✓ Loaded 18 pages

Processing: p_.pdf
  ✓ Loaded 31 pages

Processing: s_.pdf
  ✓ Loaded 33 pages

Total documents loaded: 97


In [240]:
all_pdf_documents

[Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2025-12-21T20:04:05+05:30', 'author': '', 'moddate': '2025-12-21T20:04:05+05:30', 'title': '1Intro.pmd', 'source': '..\\data\\pdfs\\i_.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1', 'source_file': 'i_.pdf', 'file_type': 'pdf'}, page_content=''),
 Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2025-12-21T20:04:05+05:30', 'author': '', 'moddate': '2025-12-21T20:04:05+05:30', 'title': '1Intro.pmd', 'source': '..\\data\\pdfs\\i_.pdf', 'total_pages': 15, 'page': 1, 'page_label': '2', 'source_file': 'i_.pdf', 'file_type': 'pdf'}, page_content=''),
 Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2025-12-21T20:04:05+05:30', 'author': '', 'moddate': '2025-12-21T20:04:05+05:30', 'title': '1Intro.pmd', 'source': '..\\data\\pdfs\\i_.pdf', 'total_pages': 15, 'page': 2, 'page_label': '3', 'source_file':

In [244]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [ ]:
chunks=split_documents(all_pdf_documents)
chunks

Split 97 documents into 79 chunks

Example chunk:
Content: Chapter 3
What Is In This Chapter?
1.	 Definition of surface water
2.	 Examples of surface water
3.	 Advantages and disadvantages of surface water
4.	 Surface water hydrology 
5.	 Raw water storage an...
Metadata: {'producer': 'Adobe PDF Library 9.0', 'creator': 'Adobe InDesign CS4 (6.0)', 'creationdate': '2011-01-19T13:42:35-09:00', 'moddate': '2011-03-17T11:14:56-08:00', 'source': '..\\data\\pdfs\\s_.pdf', 'total_pages': 33, 'page': 0, 'page_label': '1', 'source_file': 's_.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Adobe PDF Library 9.0', 'creator': 'Adobe InDesign CS4 (6.0)', 'creationdate': '2011-01-19T13:42:35-09:00', 'moddate': '2011-03-17T11:14:56-08:00', 'source': '..\\data\\pdfs\\s_.pdf', 'total_pages': 33, 'page': 0, 'page_label': '1', 'source_file': 's_.pdf', 'file_type': 'pdf'}, page_content='Chapter 3\nWhat Is In This Chapter?\n1.\t Definition of surface water\n2.\t Examples of surface water\n3.\t Advantages and disadvantages of surface water\n4.\t Surface water hydrology \n5.\t Raw water storage and flow measurements\n6.\t Surface water intake structures\n7.\t The types of pumps used to collect surface water\n8.\t Definition of groundwater\n9.\t Examples of groundwater\n10.\tAdvantages and disadvantages of groundwater\n11.\tGroundwater hydrology\n12.\tThree types of aquifers\n13.\tWell components\n14.\tData and record keeping requirements\n15.\tTransmission lines and flow meters\n16.\tGroundwater under the direct influence of surface water\nIntroductio

In [247]:
# embedding And vectorStoreDB
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        # all-MiniLM-L6-v2 is Model name in hugging face
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2
Model loaded successfully. Embedding dimension: 384


In [261]:
#VECTORE STORE DB
class VectorStore:
    """Manages document embeddings using FAISS"""

    def __init__(self, dim=384, persist_directory="../data/vector_store"):
        self.dim = dim
        self.persist_directory = persist_directory
        self.index = None
        self.documents = []
        self.metadatas = []
        self.ids = []
        self._initialize_store()

    def _initialize_store(self):
        os.makedirs(self.persist_directory, exist_ok=True)

        index_path = os.path.join(self.persist_directory, "index.faiss")
        meta_path = os.path.join(self.persist_directory, "store.pkl")

        if os.path.exists(index_path) and os.path.exists(meta_path):
            self.index = faiss.read_index(index_path)
            with open(meta_path, "rb") as f:
                data = pickle.load(f)
                self.documents = data["documents"]
                self.metadatas = data["metadatas"]
                self.ids = data["ids"]

            print("FAISS vector store loaded")
            print("Total documents:", self.index.ntotal)
        else:
            self.index = faiss.IndexFlatIP(self.dim)
            print("New FAISS vector store initialized")

    def add_documents(self, documents, embeddings):
        if len(documents) != len(embeddings):
            raise ValueError("Documents and embeddings count mismatch")

        vectors = []

        for doc, emb in zip(documents, embeddings):
            self.ids.append(f"doc_{uuid.uuid4().hex}")
            self.documents.append(doc.page_content)

            metadata = dict(doc.metadata)
            metadata["content_length"] = len(doc.page_content)
            self.metadatas.append(metadata)

            vectors.append(emb)

        vectors = np.array(vectors, dtype="float32")
        faiss.normalize_L2(vectors)
        self.index.add(vectors)

        self._persist()

        print(f"Added {len(vectors)} documents")
        print("Total documents:", self.index.ntotal)

    def similarity_search(self, query_embedding, k=5):
        if self.index.ntotal == 0:
            return []

        query_embedding = np.array([query_embedding], dtype="float32")
        faiss.normalize_L2(query_embedding)

        scores, indices = self.index.search(query_embedding, k)

        results = []
        for score, idx in zip(scores[0], indices[0]):
            if idx == -1:
                continue
            results.append({
                "content": self.documents[idx],
                "metadata": self.metadatas[idx],
                "id": self.ids[idx],
                "score": float(score)
            })

        return results

    def _persist(self):
        faiss.write_index(
            self.index,
            os.path.join(self.persist_directory, "index.faiss")
        )
        with open(os.path.join(self.persist_directory, "store.pkl"), "wb") as f:
            pickle.dump(
                {
                    "documents": self.documents,
                    "metadatas": self.metadatas,
                    "ids": self.ids,
                },
                f
            )

vectorstore=VectorStore()
vectorstore

FAISS vector store loaded
Total documents: 316


In [268]:
chunks

[Document(metadata={'producer': 'Adobe PDF Library 9.0', 'creator': 'Adobe InDesign CS4 (6.0)', 'creationdate': '2011-01-19T13:42:35-09:00', 'moddate': '2011-03-17T11:14:56-08:00', 'source': '..\\data\\pdfs\\s_.pdf', 'total_pages': 33, 'page': 0, 'page_label': '1', 'source_file': 's_.pdf', 'file_type': 'pdf'}, page_content='Chapter 3\nWhat Is In This Chapter?\n1.\t Definition of surface water\n2.\t Examples of surface water\n3.\t Advantages and disadvantages of surface water\n4.\t Surface water hydrology \n5.\t Raw water storage and flow measurements\n6.\t Surface water intake structures\n7.\t The types of pumps used to collect surface water\n8.\t Definition of groundwater\n9.\t Examples of groundwater\n10.\tAdvantages and disadvantages of groundwater\n11.\tGroundwater hydrology\n12.\tThree types of aquifers\n13.\tWell components\n14.\tData and record keeping requirements\n15.\tTransmission lines and flow meters\n16.\tGroundwater under the direct influence of surface water\nIntroductio

In [274]:
import pickle


In [275]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 79 texts...


Batches: 100%|██████████| 3/3 [00:09<00:00,  3.15s/it]

Generated embeddings with shape: (79, 384)
Added 79 documents
Total documents: 553


In [285]:
# Retriever Pipeline From VectorStore

class RAGRetriever:
    """Handles query-based retrieval from a FAISS vector store"""

    def __init__(self, vector_store, embedding_manager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0):
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        # 1. Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        query_embedding = np.array(query_embedding, dtype="float32")

        try:
            # 2. FAISS similarity search
            results = self.vector_store.similarity_search(
                query_embedding=query_embedding,
                k=top_k
            )

            # 3. Apply score threshold
            retrieved_docs = []
            for rank, r in enumerate(results, start=1):
                if r["score"] >= score_threshold:
                    retrieved_docs.append({
                        "id": r["id"],
                        "content": r["content"],
                        "metadata": r["metadata"],
                        "similarity_score": r["score"],
                        "rank": rank
                    })

            print(f"Retrieved {len(retrieved_docs)} documents")
            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []


rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [288]:
rag_retriever

In [290]:
rag_retriever.retrieve("What are the main ideas and practical uses explained in these documents?")

Retrieving documents for query: 'What are the main ideas and practical uses explained in these documents?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 12.97it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents


[{'id': 'doc_8546f48109724010921a4b3beffee60c',
  'content': 'Chapter 3\nWhat Is In This Chapter?\n1.\t Definition of surface water\n2.\t Examples of surface water\n3.\t Advantages and disadvantages of surface water\n4.\t Surface water hydrology \n5.\t Raw water storage and flow measurements\n6.\t Surface water intake structures\n7.\t The types of pumps used to collect surface water\n8.\t Definition of groundwater\n9.\t Examples of groundwater\n10.\tAdvantages and disadvantages of groundwater\n11.\tGroundwater hydrology\n12.\tThree types of aquifers\n13.\tWell components\n14.\tData and record keeping requirements\n15.\tTransmission lines and flow meters\n16.\tGroundwater under the direct influence of surface water\nIntroduction to Water Sources\nKey Words\n• Aquifer\n• Baseline Data\n• Caisson\n• Cone of Depression\n• Confined Aquifer\n• Contamination\n• Drainage Basin\n• Drawdown\n• Flume\n• Glycol \n• Groundwater\n• Impermeable\n• ntu\n• Parshall flume\n• Permeability\n• Polluted Wat

In [300]:
# RAG Pipeline- VectorDB To LLM Output Generation

import os
from dotenv import load_dotenv
load_dotenv()

# print(os.getenv("GROQ_API_KEY"))

True

In [321]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage

In [326]:
class GroqLLM:
    def __init__(self, model="llama-3.1-8b-instant"):
        if not os.environ.get("GROQ_API_KEY"):
            raise ValueError("GROQ_API_KEY not set")

        self.llm = ChatGroq(
            model=model,
            temperature=0.1
        )

    def generate_response(self, query, context):
        prompt = PromptTemplate(
            input_variables=["context", "question"],
            template="""
You are a helpful assistant.
Answer ONLY using the context below.
If not found, say "Not found in documents".

Context:
{context}

Question:
{question}

Answer:
"""
        )

        formatted_prompt = prompt.format(
            context=context,
            question=query
        )

        response = self.llm.invoke(formatted_prompt)
        return response.content


# ===============================
# Example Embedding Manager
# ===============================
class DummyEmbeddingManager:
    def generate_embeddings(self, texts):
        return np.random.rand(len(texts), 384).astype("float32")


# ===============================
# MAIN PIPELINE
# ===============================
if __name__ == "__main__":
    embedding_manager = DummyEmbeddingManager()
    vectorstore = VectorStore(dim=384)

    # Example documents
    documents = [
    Document(page_content="AI helps automate decision making in agriculture.", metadata={}),
    Document(page_content="FAISS enables fast similarity search on embeddings.", metadata={}),
    Document(page_content="Groq provides ultra-fast LLM inference.", metadata={})
    ]


    embeddings = embedding_manager.generate_embeddings(
    [doc.page_content for doc in documents]
)

    vectorstore.add_documents(documents, embeddings)

    retriever = RAGRetriever(vectorstore, embedding_manager)
    llm = GroqLLM()

    query = "What is the role of FAISS in AI systems?"

    retrieved_docs = retriever.retrieve(query)
    context = "\n\n".join(doc["content"] for doc in retrieved_docs)

    answer = llm.generate_response(query, context)
    print("\nANSWER:\n", answer)

FAISS vector store loaded
Total documents: 556
Added 3 documents
Total documents: 559
Retrieving documents for query: 'What is the role of FAISS in AI systems?'
Top K: 5, Score threshold: 0.0
Retrieved 5 documents

ANSWER:
 FAISS enables fast similarity search on embeddings.


In [328]:
# Initialize Groq LLM (FAISS-compatible, correct way)
try:
    groq_llm = GroqLLM()
    print("Groq LLM initialized successfully!")
except ValueError as e:
    print(f"Warning: {e}")
    print("Please set your GROQ_API_KEY environment variable to use the LLM.")
    groq_llm = None


Groq LLM initialized successfully!


In [329]:
rag_retriever.retrieve("Unified Multi-task Learning Framework")

Retrieving documents for query: 'Unified Multi-task Learning Framework'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  9.59it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents


[{'id': 'doc_3502defa888d4e9386af8f8f5757aa64',
  'content': 'a nephelometric turbidimeter.\n7 Water Rights – The rights, acquired \nunder the law, to use the water ac-\ncruing in surface or groundwater, for \na specified purpose in a given manner \nand usually within the limits of a given \ntime period.\nSurface Water Hydrology\n8 Drainage Basin – An area from which \nsurface runoff or groundwater re-\ncharge is carried into a single drainage \nsystem, also called a catchment area, \nwatershed, or drainage area.\n9 Watershed – A drainage basin from \nwhich surface water is obtained.\n10 Recharge Area – One from which \nprecipitation flows into the under-\nground water sources.',
  'metadata': {'producer': 'Adobe PDF Library 9.0',
   'creator': 'Adobe InDesign CS4 (6.0)',
   'creationdate': '2011-01-19T13:42:35-09:00',
   'moddate': '2011-03-17T11:14:56-08:00',
   'source': '..\\data\\pdfs\\s_.pdf',
   'total_pages': 33,
   'page': 2,
   'page_label': '3',
   'source_file': 's_.pdf',
 

In [333]:
# Integration Vectordb Context pipeline With LLM output

### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.1
)


## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [335]:
answer=rag_simple("Unified Multi-task Learning Framework",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'Unified Multi-task Learning Framework'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 12.00it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents


There is no information provided in the context about a Unified Multi-task Learning Framework. The context appears to be about water rights and surface water hydrology, and also mentions a nephelometric turbidimeter, which is a device used to measure water turbidity.


In [337]:
# Enhanced RAG Pipeline Features

# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("What are the main ideas and practical uses explained in these documents?", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'What are the main ideas and practical uses explained in these documents?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 10.19it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents


Answer: The main ideas and practical uses explained in these documents are related to water sources, specifically surface water and groundwater. The chapters cover definitions, examples, advantages, and disadvantages of both surface water and groundwater. 

Practical uses include:

1. Surface water:
   - Raw water storage and flow measurements
   - Surface water intake structures
   - Types of pumps used to collect surface water

2. Groundwater:
   - Well components
   - Data and record keeping requirements
   - Transmission lines and flow meters
   - Groundwater under the direct influence of surface water

These topics are essential for understanding water sources and their management, which is crucial for various industries, including water treatment and supply.
Sources: [{'source': 's_.pdf', 'page': 0, 'score': 0.29356399178504944, 'preview': 'Chapter 3\nWhat Is In This Chapter?\n1.\t Definition of surface water\n2.\t Examples of surface water\n3.\t Advantages and disadvantages of s

In [339]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("What are the main ideas and practical uses explained in these documents?", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'What are the main ideas and practical uses explained in these documents?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.25it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents
Streaming answer:
Use the following context to answer the question concisely.
Context:
Chapter 3
What Is In This Chapter?
1.	 Definition of surface water
2.	 Examples of surface water
3.	 Advantages and disadvantages of surface water
4.	 Surface water hydrol

ogy 
5.	 Raw water storage and flow measurements
6.	 Surface water intake structures
7.	 The types of pumps used to collect surface water
8.	 Definition of groundwater
9.	 Examples of groundwater
10.	Advantages and disadvantages of groundwater
11.	Groundwater hydrology
12.	Three types of aquifers
13.	Well components
14.	Data and record keeping requirements
15.	Transmission lines and flow meters
16.	Groundwater under the direct influence of surface water
Introduction to Water Sources
Key Words
• Aquifer
• Baseline Data
• Caisson
• Cone of Depression
• Confined Aquifer
• Contamination
• Drainage Basin
• Drawdown
• Flume
• Glycol 
• Groundwater
• Impermeable
• ntu
• Parshall flume
• Permeability
• Polluted Water
• Porosity
• Raw Water
• Recharge Area
• Riprap
• Spring
• Static Water Level
• Stratum
• Surface Runoff

Chapter 3
What Is In This Chapter?
1.	 Definition of surface water
2.	 Examples of surface water
3.	 Advantages and disadvantages of surface water
4.	 Surface water hydrology 